In [ ]:
# ! uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()


os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


## Data Ingestion


In [4]:
from langchain_community.document_loaders import TextLoader

C:\Users\SHAILENDRA\AppData\Local\Temp\ipykernel_16724\2929458509.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [5]:
loader = TextLoader("../data/AgenticAI.txt", encoding="utf8")
documents = loader.load()

In [6]:
documents[0].page_content[:500]  # Print the first 500 characters of the first documen

'The History of Agentic AI and Its Revolution\nIntroduction\n\nArtificial Intelligence has gone through several major transformations since the idea of machine intelligence first emerged in the twentieth century. What began as symbolic reasoning and rule-based systems gradually evolved into machine learning, neural networks, deep learning, and eventually large language models capable of understanding and generating human language. The latest stage of this evolution is increasingly described as Agent'

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [ ]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=150)

In [10]:
text_chunks=text_splitter.split_documents(documents)

In [11]:
text_chunks

[Document(metadata={'source': '../data/AgenticAI.txt'}, page_content='The History of Agentic AI and Its Revolution\nIntroduction'),
 Document(metadata={'source': '../data/AgenticAI.txt'}, page_content='Artificial Intelligence has gone through several major transformations since the idea of machine intelligence first emerged in the twentieth century. What began as symbolic reasoning and rule-based'),
 Document(metadata={'source': '../data/AgenticAI.txt'}, page_content='and rule-based systems gradually evolved into machine learning, neural networks, deep learning, and eventually large language models capable of understanding and generating human language. The latest'),
 Document(metadata={'source': '../data/AgenticAI.txt'}, page_content='The latest stage of this evolution is increasingly described as Agentic AI.'),
 Document(metadata={'source': '../data/AgenticAI.txt'}, page_content='Agentic AI represents a shift from AI systems that merely respond to AI systems that can reason, plan, us

In [ ]:
# ! uv pip install faiss-cpu


In [14]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [15]:
embeddings=OpenAIEmbeddings()

In [16]:
vectorstore=FAISS.from_documents(text_chunks, embeddings)

In [17]:
vectorstore

In [18]:
retriever=vectorstore.as_retriever()

In [20]:
# Perform similarity search
query = "What is the Key Characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=10)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)


Document 1:
Agentic AI represents a shift from AI systems that merely respond to AI systems that can reason, plan, use tools, execute actions, observe results, and adapt their behavior toward achieving a goal.
--------------------------------------------------
Document 2:
The latest stage of this evolution is increasingly described as Agentic AI.
--------------------------------------------------
Document 3:
Agentic AI is ultimately concerned less with whether a machine "thinks" like a human and more with whether it can behave intelligently while pursuing objectives.
--------------------------------------------------
Document 4:
Agentic AI is therefore not simply another name for a large language model. It represents an architectural and behavioral paradigm built around autonomous or semi-autonomous decision-making.
--------------------------------------------------
Document 5:
The History of Agentic AI and Its Revolution
Introduction
--------------------------------------------------


In [22]:
from langchain_core.prompts import ChatPromptTemplate
template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [23]:
prompt=ChatPromptTemplate.from_template(template)

In [24]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [25]:
from langchain_core.output_parsers import StrOutputParser

In [26]:
output_parser=StrOutputParser()

In [28]:
from langchain_openai import ChatOpenAI
llm_model=ChatOpenAI(model_name="gpt-4o-mini")

In [29]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
)

In [30]:
rag_chain.invoke("tell me about Agentic AI")

'Agentic AI represents a significant evolution in artificial intelligence, moving from systems that only respond to those capable of reasoning, planning, and executing actions autonomously or semi-autonomously. It is characterized by its ability to observe results and adapt behavior to achieve specific goals. Unlike large language models that primarily generate text, Agentic AI encompasses a broader architectural and behavioral paradigm centered around decision-making. This shift indicates a growing trend towards more autonomous AI systems that can operate effectively in complex environments. The development of Agentic AI is seen as a revolutionary stage in the field, highlighting the potential for more intelligent and adaptable AI applications.'